# Módulo AI1 · Soluciones de la práctica guiada

## Setup: corpus, BPE y modelo (celdas clave del notebook)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(torch.__version__)

corpus = [
    "la tienda abre a las nueve",
    "la tienda cierra a las nueve",
    "el pedido llega a la tienda",
    "el cliente recoge el pedido en tienda",
    "la tienda recibe pedidos cada noche",
    "el motor del negocio es el pedido",
    "el cliente pide la factura",
    "la factura llega por correo",
    "el pedido contiene bebidas",
    "la tienda vende bebidas y comida",
    "el cliente pide bebidas frias",
    "la tienda fria vende bebidas frias",
    "el pedido llega tarde",
    "la tienda abre tarde",
    "el cliente recoge tarde",
    "el motor del pedido es el cliente",
    "la tienda recibe el correo",
    "el cliente abre la tienda",
    "el pedido llega a tiempo",
    "la factura llega a la tienda",
    "el cliente pide comida fria",
    "la tienda vende comida fria",
    "el pedido llega cada noche",
    "la tienda recibe pedidos del cliente",
]
print(len(corpus), "frases")
print("ejemplo:", corpus[0])# Corpus como lista de caracteres (más un token final <eos>)
seqs = [list(s) + ["<eos>"] for s in corpus]

# Vocabulario: caracteres únicos + <eos>
vocab = sorted(set(c for s in seqs for c in s))
print("tokens iniciales:", len(vocab))

# Contar pares consecutivos en el corpus
from collections import Counter

def contar_pares(seqs):
    pares = Counter()
    for s in seqs:
        for a, b in zip(s, s[1:]):
            pares[(a, b)] += 1
    return pares

num_merges = 200
merges = []
for n in range(num_merges):
    pares = contar_pares(seqs)
    par, freq = pares.most_common(1)[0]
    if freq < 2:
        break  # nada más vale la pena fusionar
    nuevo = par[0] + par[1]
    merges.append(par)
    for i, s in enumerate(seqs):
        j = 0
        while j < len(s) - 1:
            if (s[j], s[j+1]) == par:
                s[j] = nuevo
                del s[j+1]
            j += 1

vocab = sorted(set(c for s in seqs for c in s))
print(f"tokens finales: {len(vocab)}  (con {len(merges)} fusiones BPE)")
print("algunos tokens aprendidos:", [t for t in vocab if len(t) > 1][:20])# Tokenizar de ida y vuelta (encode/decode) con el vocab y las fusiones learned
tok2id = {t: i for i, t in enumerate(vocab)}
id2tok = {i: t for t, i in tok2id.items()}

def encode_bpe(text):
    s = list(text) + ["<eos>"]
    for a, b in merges:
        j = 0
        while j < len(s) - 1:
            if (s[j], s[j+1]) == (a, b):
                s[j] = a + b
                del s[j+1]
            j += 1
    return [tok2id[t] for t in s]

def decode_bpe(ids):
    return "".join(id2tok[i] for i in ids).replace("<eos>", "")

ejemplo = encode_bpe("la tienda abre tarde")
print("IDs:", ejemplo)
print("texto:", repr(decode_bpe(ejemplo)))

In [ ]:
class AtencionCausal(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.dim, self.heads = dim, heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        #多头: cada head una "rebanada" del espacio
        H = self.heads
        q = q.view(B, T, H, C // H).transpose(1, 2)
        k = k.view(B, T, H, C // H).transpose(1, 2)
        v = v.view(B, T, H, C // H).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        att = att.masked_fill(mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v                       # (B,H,T,C/H)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = AtencionCausal(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Linear(4*dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # residual + atención
        x = x + self.mlp(self.ln2(x))     # residual + MLP
        return x

class GPT(nn.Module):
    def __init__(self, vocab_len, dim=64, heads=4, layers=2, ctx=32):
        super().__init__()
        self.ctx = ctx
        self.tok = nn.Embedding(vocab_len, dim)
        self.pos = nn.Embedding(ctx, dim)
        self.blocks = nn.Sequential(*[Bloque(dim, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_len)

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.blocks(x))

# Datos: todos los frases del corpus como una única secuencia larga
datos = []
for s in corpus:
    datos.extend(encode_bpe(s))
datos = torch.tensor(datos, dtype=torch.long)
print("secuencia total:", len(datos), "tokens; vocab:", len(vocab))

# Batch generator: corta ventanas de ctx=32
ctx = 32
def get_batch(batch=8):
    ix = torch.randint(0, len(datos) - ctx, (batch,))
    x = torch.stack([datos[i:i+ctx] for i in ix])
    y = torch.stack([datos[i+1:i+ctx+1] for i in ix])  # desplazado: siguiente token
    return x, y

modelo = GPT(len(vocab))
opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4)

paso, perdidas = 0, []
for paso in range(300):
    xb, yb = get_batch()
    logits = modelo(xb)                       # (B, T, vocab)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))
    opt.zero_grad(); loss.backward(); opt.step()
    perdidas.append(loss.item())
    if paso % 100 == 0:
        print(f"paso {paso:3d}  loss {loss.item():.3f}")
print(f"paso final: loss {loss.item():.3f} (ln(vocab)={math.log(len(vocab)):.2f} sería el azar)")

## E1 · 400 fusiones BPE

In [ ]:
# reentrenamos BPE desde cero con más fusiones
seqs2 = [list(s) + ["<eos>"] for s in corpus]
merges2 = []
for n in range(400):
    pares = contar_pares(seqs2)
    par, freq = pares.most_common(1)[0]
    if freq < 2:
        break
    merges2.append(par)
    for i, s in enumerate(seqs2):
        j = 0
        while j < len(s) - 1:
            if (s[j], s[j+1]) == par:
                s[j] = par[0]+par[1]; del s[j+1]
            j += 1
vocab2 = sorted(set(c for s in seqs2 for c in s))
print(f"tokens finales: {len(vocab2)} con {len(merges2)} fusiones (antes: 59 con 111)")

**Respuesta esperada:** con 400 intentos se paran antes (~110-130 fusiones) porque el corpus es corto. Frases repetidas quedan casi enteras como un token — exactamente el problema de BPE con corpus pequeños: memoriza frases y no generaliza. Con corpus grandes los subwords emergen sin memorizar.

## E2 · Modelo mayor (3 capas, dim=128)

In [ ]:
import time
modelo2 = GPT(len(vocab), dim=128, heads=4, layers=3, ctx=32)
opt2 = torch.optim.AdamW(modelo2.parameters(), lr=3e-4)
t0 = time.time(); losses2 = []
for paso in range(300):
    xb, yb = get_batch()
    logits = modelo2(xb)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))
    opt2.zero_grad(); loss.backward(); opt2.step()
    losses2.append(loss.item())
print(f"loss final {loss.item():.3f} en {time.time()-t0:.1f}s (el base: ~0.1 en ~5s)")

**Esperado:** loss baja más (más capacidad) pero tarda ~3-5× más en CPU. Con corpus tan pequeño la mejora se satura rápido — la escala solo rinde con más datos (E5 con gpt2).

## E3 · Sin máscara causal

In [ ]:
# setup de la atención manual (misma matriz E y pesos fijos del notebook)
d, T = 8, 6
E = np.random.default_rng(42).normal(size=(T, d))
Wq, Wk, Wv = (np.random.default_rng(i+1).normal(size=(d, d)) * 0.3 for i in range(3))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def atencion_libre_np(E, Wq, Wk, Wv):
    T, d = E.shape
    Q, K, V = E @ Wq, E @ Wk, E @ Wv
    A = softmax((Q @ K.T) / math.sqrt(d))
    return A, A @ V
A2, _ = atencion_libre_np(E, Wq, Wk, Wv)
print(A2[0])   # toda la fila tiene pesos no nulos

**Respuesta:** sin máscara, todo token atiende a todos. Eso rompe la tarea del siguiente token: al predecir t{i+1}, el modelo ya estaría mirando t{i+1} (leaks del futuro). Es la diferencia GPT (causal) vs BERT (bidireccional).

## E4 · Temperaturas

In [ ]:
# setup de generación: helper idéntico al notebook + semilla robusta
@torch.no_grad()
def generar(modelo, ids_semilla, max_nuevos=30, temperature=1.0):
    ids = list(ids_semilla)
    for _ in range(max_nuevos):
        contexto = torch.tensor([ids[-modelo.ctx:]], dtype=torch.long)
        logits = modelo(contexto)[0, -1]
        probs = F.softmax(logits / temperature, dim=-1)
        nuevo = torch.multinomial(probs, 1).item()
        if nuevo == tok2id["<eos>"]:
            break
        ids.append(nuevo)
    return ids

semilla = "la tienda "
ids_semilla = [i for i in encode_bpe(semilla) if "<eos>" not in id2tok[i]]
if not ids_semilla:
    ids_semilla = [tok2id[c] for c in list(semilla)[:2]]
print("semilla:", decode_bpe(ids_semilla))

grupos = {}
for temp in (0.25, 1.8):
    outs = [decode_bpe(generar(modelo, ids_semilla, max_nuevos=20, temperature=temp) + [tok2id["<eos>"]]) for _ in range(10)]
    grupos[temp] = outs
    print(f"temp={temp}: {len(set(outs))} frases distintas de 10")

**Contar y clasificar:** con temp 0.25 casi todo sale (a): copias literales del corpus; con 1.8 mezclas de (b) y (c). La temperatura divide los logits: **baja** => distribution "puntual" (repite lo más probable); **sube** => aplana (improvisa). No cambia qué sabe el modelo, solo cómo elige.

## E5 · gpt2 con Hugging Face (Colab GPU)

In [ ]:
# Ejecutar en Colab (GPU recomendado): pip install transformers
# from transformers import AutoTokenizer, AutoModelForCausalLM
# tok = AutoTokenizer.from_pretrained("gpt2")
# modelo_hf = AutoModelForCausalLM.from_pretrained("gpt2")
# x = tok("la tienda abre a las", return_tensors="pt")
# for t in (0.5, 1.0):
#     y = modelo_hf.generate(**x, do_sample=True, temperature=t, max_new_tokens=20)
#     print(tok.decode(y[0]))
print("esqueleto listo para ejecutar en Colab")

**Comparación clave:** gpt2 (124M, WebText) genera frases coherentes fuera del corpus; tu GPT de 70k solo recombina el corpus. El diagrama es idéntico: cambia la escala de datos, parámetros y cómputo — la lección de «Scaling».